## 1. Recurrent Neural network example 
for more detials check the pdf of this session
https://github.com/sumaia-a-k/NLP-2026/wiki/RNN

In [2]:
!pip install tensorflow

  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:-

------------------

In [3]:
import pandas as pd
import numpy as np
import re  
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding

### TensorFlow is a free, open-source library from Google that’s designed to make building and training machine learning models easier

#### 1.pad_sequences
What it does:
Converts a list of variable-length sequences (e.g., sentences) into a single 2D tensor of uniform length by adding zeros (padding) or cutting off long sequences (truncating).
Why we need it:
RNNs process sequences step-by-step, so they natively handle different lengths. However, GPUs/TPUs and Keras require rectangular tensors for efficient batch processing. pad_sequences solves the batching problem.

from tensorflow.keras.preprocessing.sequence import pad_sequences

pad_sequences(sequences, maxlen=None, padding='post', truncating='post', value=0)

maxlen: Target length. Shorter sequences get padded; longer ones get truncated.
padding: 'pre' (zeros at start) or 'post' (zeros at end).
         'pre' is standard for Many-to-One RNNs so the final hidden state captures the active memory of the final words, rather than fading out over trailing zeros.
         
value: Padding token (usually 0).
Use code with caution.With these code components defined, your students now have the complete blueprint. Would you like to wrap up by putting these 5 components into a complete 10-line Python code block showing how a model is compiled and trained, so they can copy-paste it as a lab assignment?
value: Padding token (usually 0).

#### 2.sequential 

A Keras model container that stacks layers linearly, where the output of one layer becomes the input of the next.

Why we need it:
It’s the simplest way to define a neural network architecture. For RNN tasks, you typically build:
Input → Embedding → SimpleRNN → Dense → Output

#### 3.Embedding 

What it does:
A trainable lookup table that converts integer word indices into dense, continuous vectors (embeddings).

Embedding(input_dim=vocab_size, output_dim=embedding_dim)

input_dim: Size of vocabulary (e.g., 5000)
output_dim: Dimension of each word vector (e.g., 64, 100, 300)

#### 4.SimpleRnn

Keras' implementaion of the vanilla RNN 

SimpleRNN(units=hidden_size, return_sequences=False, return_state=False)

return_sequences: 

    False (default) → returns only the final hidden state h_T (used for sequence-level tasks like sentiment)
    True → returns hidden states at every timestep (used for token-level tasks like POS tagging or language modeling)

return_state: Advanced use; returns h_T alongside outputs

#### 5.dense 

A standard fully-connected neural network layer. Every input connects to every output neuron.

Maps the RNN’s hidden representation to your desired output format (e.g., 2 classes for positive/negative sentiment).

Dense(units=output_classes, activation='softmax')

units: Number of output neurons (e.g., 2 for binary sentiment)
activation: 

    'softmax' → multi-class probability distribution
    'sigmoid' → binary classification
    'linear' → regression

------

In [8]:
data = pd.read_csv('swiggy.csv')
print("Columns in the dataset:")
print(data.columns.tolist())

Columns in the dataset:
['ID', 'Area', 'City', 'Restaurant Price', 'Avg Rating', 'Total Rating', 'Food Item', 'Food Type', 'Delivery Time', 'Review']


--------

In [9]:
data.head()

,ID,Area,City,Restaurant Price,Avg Rating,Total Rating,Food Item,Food Type,Delivery Time,Review
0,1,Suburb,Ahmedabad,600,4.2,6198,Sushi,Fast Food,30-40 min,"Good, but nothing extraordinary."
1,2,Business District,Pune,200,4.7,4865,Pepperoni Pizza,Non-Vegetarian,50-60 min,"Good, but nothing extraordinary."
2,3,Suburb,Bangalore,600,4.7,2095,Waffles,Fast Food,50-60 min,Late delivery ruined it.
3,4,Business District,Mumbai,900,4.0,6639,Sushi,Vegetarian,50-60 min,Best meal I've had in a while!
4,5,Tech Park,Mumbai,200,4.7,6926,Spring Rolls,Gluten-Free,20-30 min,Mediocre experience.


#### Text Cleaning
- Converts review text to lowercase
- Removes special characters and punctuation
- Creates sentiment labels from ratings
- Removes rows with missing values

In [10]:
data["Review"] = data["Review"].str.lower()
data["Review"] = data["Review"].replace(r'[^a-z0-9\s]', '', regex=True)

data['sentiment'] = data['Avg Rating'].apply(lambda x: 1 if x > 3.5 else 0)
data = data.dropna()

data.head()

,ID,Area,City,Restaurant Price,Avg Rating,Total Rating,Food Item,Food Type,Delivery Time,Review,sentiment
0,1,Suburb,Ahmedabad,600,4.2,6198,Sushi,Fast Food,30-40 min,good but nothing extraordinary,1
1,2,Business District,Pune,200,4.7,4865,Pepperoni Pizza,Non-Vegetarian,50-60 min,good but nothing extraordinary,1
2,3,Suburb,Bangalore,600,4.7,2095,Waffles,Fast Food,50-60 min,late delivery ruined it,1
3,4,Business District,Mumbai,900,4.0,6639,Sushi,Vegetarian,50-60 min,best meal ive had in a while,1
4,5,Tech Park,Mumbai,200,4.7,6926,Spring Rolls,Gluten-Free,20-30 min,mediocre experience,1


------

#### Tokenization and Padding

The text data is converted into numerical sequences and padded to ensure all inputs have the same length for model training.

- max_features = 5000
Keep only the 5,000 most frequent words in the entire dataset. Rare words are ignored

- max_length = 200
Standardize every review to exactly 200 tokens. Shorter reviews get padded; longer ones get cut.

- tokenizer = Tokenizer(num_words=max_features)
Creates a Keras tokenizer that will build a word → index dictionary. num_words caps the dictionary size.
index 0 is reserved for padding

- tokenizer.fit_on_texts
Scans all reviews, counts word frequencies, and builds the internal vocabulary map (tokenizer.word_index).

- y = data['sentiment'].values
Extracts the target labels (e.g., 0 for negative, 1 for positive) for supervised training.

- tokenizer.texts_to_sequences(data["Review"])
Converts each review string into a list of integers using the learned dictionary.


In [15]:
max_features = 5000
max_length = 200

tokenizer = Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(data["Review"])
print(tokenizer.index_word)
print(len(tokenizer.word_index))

{1: 'and', 2: 'taste', 3: 'it', 4: 'meal', 5: 'a', 6: 'worth', 7: 'the', 8: 'price', 9: 'delivery', 10: 'but', 11: 'nothing', 12: 'perfectly', 13: 'cooked', 14: 'wellseasoned', 15: 'tasty', 16: 'highly', 17: 'recommended', 18: 'superb', 19: 'packaging', 20: 'presentation', 21: 'my', 22: 'new', 23: 'favorite', 24: 'dish', 25: 'order', 26: 'again', 27: 'amazing', 28: 'quick', 29: 'experience', 30: 'delicious', 31: 'fresh', 32: 'best', 33: 'ive', 34: 'had', 35: 'in', 36: 'while', 37: 'absolutely', 38: 'loved', 39: 'not', 40: 'special', 41: 'edible', 42: 'decent', 43: 'neither', 44: 'great', 45: 'nor', 46: 'bad', 47: 'good', 48: 'extraordinary', 49: 'would', 50: 'if', 51: 'needed', 52: 'standard', 53: 'quality', 54: 'average', 55: 'was', 56: 'okay', 57: 'mediocre', 58: 'too', 59: 'spicy', 60: 'oily', 61: 'disappointed', 62: 'worst', 63: 'ever', 64: 'wouldnt', 65: 'as', 66: 'described', 67: 'cold', 68: 'stale', 69: 'food', 70: 'terrible', 71: 'late', 72: 'ruined'}
72


In [16]:
X = pad_sequences(tokenizer.texts_to_sequences(
    data["Review"]), maxlen=max_length)
y = data['sentiment'].values

-------

The stratify parameter in scikit-learn's train_test_split ensures that the class distribution in the resulting splits mirrors the class distribution in the original dataset. It is primarily used in classification tasks to prevent accidental class imbalance between training, validation, and test sets.

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

-------

 Building RNN Model

A simple RNN model is built and compiled for binary sentiment classification.

    Sequential([...]) creates the neural network model
    Embedding(...) converts words into dense vector representations
    SimpleRNN(64, activation='tanh') adds an RNN layer with 64 units
    Dense(1, activation='sigmoid') creates the binary output layer
    model.compile(...) configures the model with loss function, optimizer, and accuracy metric

In [21]:
model = Sequential([
    Embedding(input_dim=max_features, output_dim=16),
    SimpleRNN(64, activation='tanh', return_sequences=False),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [22]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=1
)

score = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {score[1]:.2f}")

Epoch 1/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 9s 35ms/step - accuracy: 0.7073 - loss: 0.6071 - val_accuracy: 0.7156 - val_loss: 0.6014
Epoch 2/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.7160 - loss: 0.5987 - val_accuracy: 0.7156 - val_loss: 0.5976
Epoch 3/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - accuracy: 0.7160 - loss: 0.5978 - val_accuracy: 0.7156 - val_loss: 0.5957
Epoch 4/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.7160 - loss: 0.5967 - val_accuracy: 0.7156 - val_loss: 0.5952
Epoch 5/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.7160 - loss: 0.5970 - val_accuracy: 0.7156 - val_loss: 0.5967
Test accuracy: 0.72


--------

In [20]:
def predict_sentiment(review_text):
    text = review_text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)

    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_length)

    prediction = model.predict(padded)[0][0]
    return f"{'Positive' if prediction >= 0.5 else 'Negative'} (Probability: {prediction:.2f})"


sample_review = "The food was great."
print(f"Review: {sample_review}")
print(f"Sentiment: {predict_sentiment(sample_review)}")

Review: The food was great.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step
Sentiment: Positive (Probability: 0.71)
